In [11]:
import pprint

import requests
from bs4 import BeautifulSoup
import pycountry

base_url = "https://download.geofabrik.de/"

countries = {
    "continents": [
        {
            "name": "europe",
            "countries": [
                "albania",
                "andorra",
                "austria",
                "belarus",
                "belgium",
                "bosnia-herzegovina",
                "bulgaria",
                "croatia",
                "cyprus",
                "czech-republic",
                "denmark",
                "estonia",
                "faroe-islands",
                "finland",
                "france",
                "georgia",
                "germany",
                "greece",
                "guernsey-jersey",
                "hungary",
                "iceland",
                "ireland-and-northern-ireland",
                "isle-of-man",
                "italy",
                "kosovo",
                "latvia",
                "latvia",
                "liechtenstein",
                "lithuania",
                "luxembourg",
                "macedonia",
                "malta",
                "moldova",
                "monaco",
                "montenegro",
                "netherlands",
                "norway",
                "poland",
                "portugal",
                "romania",
                "serbia",
                "slovakia",
                "slovenia",
                "spain",
                "sweden",
                "switzerland",
                "turkey",
                "ukraine",
                "united-kingdom",
            ],
        },
        {
            "name": "north-america",
            "countries": ["us", "canada", "mexico", "greenland"],
        },
        {
            "name": "australia-oceania",
            "countries": ["new-zealand"],
        },
    ]
}

In [12]:

workflows = []
for continent in countries["continents"]:
    for country in continent["countries"]:
        print(continent["name"], country)
        
        url = base_url + continent["name"] + "/"

        response = requests.get(url + country + ".html")
        soup = BeautifulSoup(response.text, "html.parser")

        maps = []

        subregions = False
        try:
            table = soup.find_all("table", id="subregions")[1]
            rows = table.find_all("tr")
            state_cnt = 1
            for row in rows:
                for entry in row.find_all('a', href=True):
                    link = entry['href']
                    if "latest.osm.pbf" in link:
                        name = link.split('/')[-1].replace('-latest.osm.pbf', '')
                        #print("{ name:", name,", url: ",base_url + link, ", state: \"" + str(state_cnt).rjust(2, '0') + "00\"},")
                        maps.append(
                            {
                                "name": name,
                                "url": url + link,
                                "state": str(state_cnt).rjust(2, '0') + "00"
                            }
                        )
                        state_cnt += 1
            subregions = True
        except IndexError:
            pass
        
        if not subregions:
            try:
                for link in soup.find_all('a', href=True):
                    if("latest.osm.pbf" in link["href"]):
                        #print("{ name:", country,", url: ",url + link['href'], ", state: \"" + str(1).rjust(2, '0') + "00\"},")
                        maps.append(
                            {
                                "name": country,
                                "url": url + link['href'],
                                "state": '0000'
                            }
                        )
                subregions = True
            except IndexError:
                pass


        # three tries to get the country code...
        countrycode = ""
        try:
            countrycode = pycountry.countries.search_fuzzy(country)[0].alpha_2
        except LookupError:
            pass
        
        if countrycode == "":
                try:
                    countrycode = pycountry.countries.search_fuzzy(country.replace("-"," "))[0].alpha_2
                except LookupError:
                    pass
        
        if countrycode == "":
                try:
                    countrycode = pycountry.countries.search_fuzzy(country.split("-")[0])[0].alpha_2
                except LookupError:
                    pass

        if countrycode == "":
            #manual mapping for the countries that still don't work
            if country == "turkey":
                countrycode = "TR"
            else:
                print("Country code not found for", country)

        

        workflows.append(
            {
                "name": country,
                "maps": maps,
                "countrycode": countrycode
            }
        )


europe albania
europe andorra
europe austria
europe belarus
europe belgium
europe bosnia-herzegovina
europe bulgaria
europe croatia
europe cyprus
europe czech-republic
europe denmark
europe estonia
europe faroe-islands
europe finland
europe france
europe georgia
europe germany
europe greece
europe guernsey-jersey
europe hungary
europe iceland
europe ireland-and-northern-ireland
europe isle-of-man
europe italy
europe kosovo
europe latvia
europe latvia
europe liechtenstein
europe lithuania
europe luxembourg
europe macedonia
europe malta
europe moldova
europe monaco
europe montenegro
europe netherlands
europe norway
europe poland
europe portugal
europe romania
europe serbia
europe slovakia
europe slovenia
europe spain
europe sweden
europe switzerland
europe turkey
europe ukraine
europe united-kingdom
north-america us
north-america canada
north-america mexico
north-america greenland


In [13]:
print(pycountry.countries.search_fuzzy("Türkiye")[0].alpha_2)

TR


In [14]:
# write database to files DO NOT USE YAML lib

for entry in workflows:
    
    if 'maps' in entry.keys():
        src = open('template_state_country.yml', 'r')
        dst = open('.github/workflows/build-' + entry['name'] + '.yml', 'w')
        for index, line in enumerate(src):
            if index == 0:
                dst.write('name: ' + entry['name'] + '\n')
            elif index == 3:
                dst.write('  countrycode: ' + entry['countrycode'] + '\n')
            elif "maps:" in line:
                dst.write(line)
                dst.write('          [\n')
                for map in entry['maps']:
                    dst.write('            {name: ' + map['name'] + ', url: ' + map['url'] + ', state: "' + map['state'] +'"},\n')
                dst.write('          ]\n')
            else:
                dst.write(line)
        dst.close()


In [15]:
# create status badges
for entry in workflows:
    print("[![", end='')
    print(entry['name'], end='')
    print("](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-", end='')
    print(entry['name'], end='')
    print(".yml/badge.svg)](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-", end='')
    print(entry['name'], end='')
    print(".yml)")


[![albania](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-albania.yml/badge.svg)](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-albania.yml)
[![andorra](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-andorra.yml/badge.svg)](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-andorra.yml)
[![austria](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-austria.yml/badge.svg)](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-austria.yml)
[![belarus](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-belarus.yml/badge.svg)](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-belarus.yml)
[![belgium](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-belgium.yml/badge.svg)](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-belgium.yml)
[![bosnia-herzegovina](https://github.com/manujedi/BSC300-Maps/actions/workflows/build-bosnia-herzegovina.yml/badge